In [11]:
%matplotlib qt
import os
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
from mne.io import read_raw_brainvision, read_raw_fif

from scipy.signal import find_peaks
import numpy as np
from mne.channels import read_dig_captrak, make_standard_montage
from mne.preprocessing import ICA
from mne_icalabel import label_components
import customtkinter as ctk
from mne.coreg import Coregistration
from mne import (open_report, events_from_annotations, Epochs, write_trans,
                setup_source_space, make_bem_model, make_bem_solution, make_forward_solution, compute_covariance,
                extract_label_time_course, read_labels_from_annot)
from mne.minimum_norm import make_inverse_operator, apply_inverse
from mne.datasets import fetch_fsaverage
from scipy.signal import find_peaks
import warnings
from mne.preprocessing import ICA, EOGRegression, create_eog_epochs, create_ecg_epochs, compute_proj_ecg, compute_proj_eog, EOGRegression
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
#on which system the code should run
local_config = 'Beat' #Please change according to your needs
data_source = 'Regensburg'

In [13]:
from pathlib import Path
if local_config=='Beat':
    data_dir = Path('/home/toedtli/Tinnitus/Data')
elif local_config=='Payam':
    data_dir = Path('/Users/payamsadeghishabestari/antinomics_clean_codes/subjects/')
elif local_config=='Milena':
    data_dir = Path('/Users/milenaengelke/tmp/')
elif local_config=='Nathan':
    raise NotImplementedError()
elif local_config=='Shagun':
    raise NotImplementedError()
else:
    raise NotImplementedError()

In [14]:
if data_source == 'Regensburg':
    #Milena
    subjects_dir = data_dir / 'data_regensburg'
    subject_id = '50050'
    subject_id = '50001'
    paradigm = 'oddball'
    site="Regensburg"
elif data_source == 'Illinois':
    #Shagun
    subjects_dir = data_dir / 'data_illinois/subjects'
    subject_id = '40015'
    paradigm = 'oddball'
    site="Illinois"
elif data_source == 'Dublin':
    #Nathan
    subjects_dir = data_dir / 'data_dublin/subjects'
    subject_id = '20035'
    paradigm = 'oddball'
    site="Dublin"

In [15]:
%matplotlib qt
from eeg.eeg_preprocessing_Zurich import preprocessing
preprocessing(subject_id=subject_id,
				site=site,
                subjects_dir=subjects_dir,
                paradigm=paradigm,
                manual_data_scroll=False,
                run_ica=False, # still throws an error on Dublin data 
                manual_ica_removal=False, 
                ssp_eog=False,
                ssp_ecg=False,
                create_report=True,
                saving_dir=None,
                verbose="ERROR")

 20%|█████████                                    

Loading raw EEG data ...



 40%|██████████████████                           

Resampling, filtering and re-referencing ...



 60%|███████████████████████████                  

Creating report and saving...



 80%|████████████████████████████████████         

EEG data were preprocessed sucessfully!



In [18]:
#debugging Nathan:
if data_source == 'Dublin':
    fname = Path('/home/toedtli/Tinnitus/Data/data_dublin/subjects/20035/EEG/oddball/raw_prep.fif')
    raw = read_raw_fif(fname, preload=True)
    import mne
    events = mne.find_events(raw)
    #events, _ = events_from_annotations(raw)
    print(events,np.unique(events[:,2]))

In [19]:
%matplotlib qt
from eeg.eeg_processing_Zurich import run_rs_analysis

run_rs_analysis(
        subject_id,
        subjects_dir=subjects_dir,
        visit=1,
        event_ids=None,
        source_analysis=True,
        mri=False,
        subjects_fs_dir=None,
        manual_data_scroll=False,
        create_report=True,
        saving_dir=None,
        overwrite = True,
        verbose="ERROR"
        )

 38%|████████████████▉                            

Loading preprocessed EEG data ...

Creating epochs...

This recording is only eyes open or eyes closed.
Loading MRI information of Freesurfer template subject ...

Computing forward solution ...



 62%|████████████████████████████▏                

Using ad hoc noise covariance for the recording ...

Computing the minimum-norm inverse solution ...



 88%|███████████████████████████████████████▍     

Creating report...

True
True


100%|█████████████████████████████████████████████

Analysis finished successfully!



In [20]:
%matplotlib qt
from eeg.eeg_processing import run_erp_analysis

run_erp_analysis(
        subject_id=subject_id,
        subjects_dir=subjects_dir,
        paradigm=paradigm,
        session="omi",
        events=None,
        source_analysis=True,
        mri=False,
        subjects_fs_dir=None,
        manual_data_scroll=False,
        create_report=True,
        saving_dir=None,
        overwrite=True,
        verbose="ERROR"
        )

 11%|█████                                        

Loading preprocessed EEG data ...



 22%|██████████                                   

Creating epochs...

Computing Evoked objects and saving it...



FileNotFoundError: [Errno 2] Datei oder Verzeichnis nicht gefunden: '/home/toedtli/Tinnitus/Data/data_regensburg/50001/EEG/oddball/omi/New Segment/LostSamples: 1-evo.fif'

In [ ]:
sys.exit(0) # code further below does not run generally

Eyetracking

In [ ]:
import mne
raw = mne.io.read_raw_brainvision("/Users/payamsadeghishabestari/antinomics_clean_codes/subjects/zdfy/EEG/xxxxx/zdfy_xxxxx.vhdr")
mne.read_events(raw)

In [ ]:
raw.annotations

In [ ]:
from mne.io import read_raw_brainvision, read_raw_eyelink
from mne import events_from_annotations
from mne.preprocessing import realign_raw


## load
fname_eeg = "/Users/payamsadeghishabestari/Downloads/alertness_01_H036.vhdr"
fname_et = "/Users/payamsadeghishabestari/Downloads/local_al1H036.asc"

raw_eeg = read_raw_brainvision(fname_eeg, preload=True)
raw_eye = read_raw_eyelink(fname=fname_et, create_annotations=True) 
events_eeg, event_ids_eeg = events_from_annotations(raw_eeg)
events_eye, event_ids_eye = events_from_annotations(raw_eye)

## for now
stim_id_eeg_1, stim_id_eeg_2 = 8, 9
stim_id_eye_1, stim_id_eye_2 = 4, 5

s_raw = events_eeg[(events_eeg[:,2] == stim_id_eeg_1) | (events_eeg[:,2] == stim_id_eeg_2)][:, 0]
s_other = events_eye[(events_eye[:,2] == stim_id_eye_1) | (events_eye[:,2] == stim_id_eye_2)][:, 0]

## realigning
realign_raw(raw=raw_eeg,
            other=raw_eye,
            t_raw=s_raw / raw_eeg.info["sfreq"] - raw_eeg.first_time,
            t_other=s_other / raw_eye.info["sfreq"] - raw_eye.first_time,
            verbose=None)

raw_eye.add_channels([raw_eeg], force_update_info=True)
events, event_dict = events_from_annotations(raw_eye)
del raw_eeg  

In [ ]:
fname_eeg = "/Users/payamsadeghishabestari/Downloads/alertness_01_H036.vhdr"
fname_et = "/Users/payamsadeghishabestari/Downloads/local_al1H036.asc"

raw_eeg = read_raw_brainvision(fname_eeg, preload=True)
raw_eye = read_raw_eyelink(fname=fname_et, create_annotations=True) 

In [ ]:
raw_eye.plot()

In [ ]:
raw_eye.plot()